# COVID-Only vs UKR-Only vs COVID+UKR Merged

Compares the independently trained COVID-only and UKR-only checkpoints against the merged COVID+UKR checkpoint across the evaluated graph/task suite.

The notebook reads `covid_vs_ukr_vs_merged_eval_results.csv`, filters the merged model to one checkpoint by default, and plots a dataset/task grid with shots on the x-axis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CSV_NAME = "covid_vs_ukr_vs_merged_eval_results.csv"
CSV_PATH = Path(CSV_NAME)
if not CSV_PATH.exists():
    CSV_PATH = Path("scripts/plotting/covid_vs_ukr_vs_merged") / CSV_NAME
CSV_PATH = CSV_PATH.resolve()

# Swap these when needed.
SPLIT = "test"
METRIC = "roc_auc"

# Merged eval CSV contains both 10k and 60k checkpoints. Use one for the
# clean three-way comparison: COVID-only, UKR-only, merged.
MERGED_CHECKPOINT = "60k"

SHOT_ORDER = [0, 3, 10]
DATASET_ORDER = [
    "covid19_twitter",
    "ukr_rus_twitter",
    "midterm",
    "covid_political",
    "election2020",
    "ukr_rus_suspended",
]
TASK_ORDER = ["nm", "lp", "pl"]
TASK_LABELS = {
    "nm": "Neighbor matching",
    "lp": "Temporal link prediction",
    "pl": "Classification",
}
TRAINING_ORDER = ["covid_only", "ukr_only", "merged_covid_ukr"]
TRAINING_LABELS = {
    "covid_only": "COVID-only",
    "ukr_only": "UKR-only",
    "merged_covid_ukr": f"COVID+UKR merged ({MERGED_CHECKPOINT})",
}
METRICS = ["accuracy", "f1", "roc_auc"]

In [ ]:
raw = pd.read_csv(CSV_PATH)
raw["shots"] = raw["shots"].astype(int)

for col in METRICS:
    if col in raw.columns:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

required = {"training_graph", "training_label", "checkpoint_label"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"Combined CSV is missing columns: {sorted(missing)}")

unexpected_training = sorted(set(raw["training_graph"]) - set(TRAINING_ORDER))
unexpected_shots = sorted(set(raw["shots"]) - set(SHOT_ORDER))
if unexpected_training:
    raise ValueError(f"Unexpected training_graph values: {unexpected_training}")
if unexpected_shots:
    raise ValueError(f"Unexpected shot values: {unexpected_shots}")

single_dates = set(raw.loc[raw["training_graph"].isin(["covid_only", "ukr_only"]), "timestamp"].astype(str).str[:10])
if single_dates != {"15_06_2026"}:
    raise ValueError(f"COVID-only/UKR-only rows are not restricted to 15_06_2026: {sorted(single_dates)}")

df = raw[
    (raw["training_graph"] != "merged_covid_ukr")
    | (raw["checkpoint_label"].astype(str) == MERGED_CHECKPOINT)
].copy()

df["training_label"] = df["training_graph"].map(TRAINING_LABELS)
df["dataset"] = pd.Categorical(df["dataset"], categories=DATASET_ORDER, ordered=True)
df["task"] = pd.Categorical(df["task"], categories=TASK_ORDER, ordered=True)
df["training_graph"] = pd.Categorical(df["training_graph"], categories=TRAINING_ORDER, ordered=True)
df["task_label"] = df["task"].astype(str).map(TASK_LABELS).fillna(df["task"].astype(str))
df = df.sort_values(["split", "dataset", "task", "training_graph", "shots"])

print(f"reading {CSV_PATH}")
print(f"raw rows: {len(raw)}; filtered rows: {len(df)}")
display(df.groupby(["training_graph", "shots"], observed=True).size().rename("rows").reset_index())
display(df.groupby(["dataset", "task", "shots"], observed=True).size().rename("models_present").reset_index().head(20))
display(df.head())

In [ ]:
def plot_metric_grid(data, *, split=SPLIT, metric=METRIC, datasets=DATASET_ORDER, tasks=TASK_ORDER):
    subset = data[(data["split"] == split) & data[metric].notna()].copy()
    if subset.empty:
        raise ValueError(f"No rows for split={split!r}, metric={metric!r}")

    palette = dict(zip(TRAINING_ORDER, sns.color_palette("Set2", n_colors=len(TRAINING_ORDER))))

    fig, axes = plt.subplots(
        nrows=len(datasets),
        ncols=len(tasks),
        figsize=(4.9 * len(tasks), 2.7 * len(datasets)),
        sharex=True,
        sharey=True,
    )

    if len(datasets) == 1 and len(tasks) == 1:
        axes = [[axes]]
    elif len(datasets) == 1:
        axes = [axes]
    elif len(tasks) == 1:
        axes = [[ax] for ax in axes]

    for row_idx, dataset in enumerate(datasets):
        for col_idx, task in enumerate(tasks):
            ax = axes[row_idx][col_idx]
            panel = subset[(subset["dataset"].astype(str) == dataset) & (subset["task"].astype(str) == task)]
            task_label = TASK_LABELS.get(task, task)
            ax.set_title(f"{dataset}\n{task_label}", fontsize=10)

            if panel.empty:
                ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center", va="center", color="0.45")
            else:
                for training_graph in TRAINING_ORDER:
                    line = panel[panel["training_graph"].astype(str) == training_graph].sort_values("shots")
                    if line.empty:
                        continue
                    ax.plot(
                        line["shots"],
                        line[metric],
                        marker="o",
                        linewidth=2,
                        markersize=5,
                        label=TRAINING_LABELS.get(training_graph, training_graph),
                        color=palette[training_graph],
                    )

            ax.set_ylim(0, 1.02)
            ax.set_xticks(SHOT_ORDER)
            ax.grid(True, alpha=0.3)
            if row_idx == len(datasets) - 1:
                ax.set_xlabel("shots")
            else:
                ax.set_xlabel("")
            if col_idx == 0:
                ax.set_ylabel(metric)
            else:
                ax.set_ylabel("")

    handles, labels = [], []
    for ax in fig.axes:
        h, l = ax.get_legend_handles_labels()
        for handle, label in zip(h, l):
            if label not in labels:
                handles.append(handle)
                labels.append(label)

    if handles:
        fig.legend(handles, labels, title="training graph", loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.01))

    fig.suptitle(f"{split} {metric}: COVID-only vs UKR-only vs merged", y=1.03, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_metric_grid(df, split=SPLIT, metric=METRIC)

In [ ]:
# Side-by-side metric table. Missing cells usually mean that eval did not run,
# failed, or was not supported for that graph/task.
table = (
    df[df["split"] == SPLIT]
    .pivot_table(
        index=["dataset", "task", "shots"],
        columns="training_label",
        values=METRIC,
        observed=True,
    )
    .reset_index()
    .sort_values(["dataset", "task", "shots"])
)

cols = [TRAINING_LABELS[key] for key in TRAINING_ORDER]
available = [col for col in cols if col in table.columns]
merged_col = TRAINING_LABELS["merged_covid_ukr"]
if merged_col in table.columns:
    if "COVID-only" in table.columns:
        table["merged - COVID-only"] = table[merged_col] - table["COVID-only"]
    if "UKR-only" in table.columns:
        table["merged - UKR-only"] = table[merged_col] - table["UKR-only"]

display(table)

In [ ]:
# Coverage diagnostics for the selected merged checkpoint.
coverage = (
    df.groupby(["dataset", "task", "shots", "training_label"], observed=True)
      .size()
      .rename("rows")
      .reset_index()
      .pivot_table(
          index=["dataset", "task", "shots"],
          columns="training_label",
          values="rows",
          fill_value=0,
          observed=True,
      )
      .reset_index()
      .sort_values(["dataset", "task", "shots"])
)

display(coverage)
display(coverage[(coverage[[col for col in coverage.columns if col not in {"dataset", "task", "shots"}]] == 0).any(axis=1)])

In [ ]:
# Long-form table for export/copying.
long_table = (
    df[df["split"] == SPLIT]
    [["dataset", "task", "training_label", "shots", METRIC, "run_name"]]
    .sort_values(["dataset", "task", "training_label", "shots"])
)
display(long_table)